# Notebook 4 — Embed Wikipedia Articles into Qdrant

This is the **fourth step** of the GraphRAG pipeline. It splits each Wikipedia article into overlapping text chunks, embeds them with **Azure OpenAI `text-embedding-3-small`**, and stores the resulting 1 536-dimensional vectors in a **Qdrant** collection.

## What this notebook does
1. Configures Azure OpenAI and Qdrant settings
2. (Re)creates the `wikipedia_scientists` Qdrant collection with cosine similarity
3. Splits each article into overlapping chunks (~1 500 characters, 150-character overlap), preferring paragraph boundaries
4. Calls the Azure OpenAI embedding API for each chunk and caches the result to `data/wikipedia_vectors/`
5. Upserts all vectors to Qdrant in batches

## Why overlapping chunks?
Overlapping ensures that sentences near a chunk boundary are not split in a way that loses context. The 150-character overlap means the start of each chunk repeats the end of the previous one.

## Qdrant payload
Each vector point stores the following metadata alongside the embedding:
- `filename` — source `.txt` file
- `title` — scientist name
- `chunk_index` / `chunk_total` — position in the article
- `text` — the raw chunk text (needed to build the RAG context)

## Caching
Embeddings are expensive to compute. Each file's vectors are saved to `data/wikipedia_vectors/<name>.json` on first run and loaded from disk on subsequent runs.

## Output
A populated Qdrant collection (`wikipedia_scientists`) with ~300–400 vector points ready for semantic search.

## Next step
Run **Notebook 5** to query the knowledge base using Vector-only, Graph-only, and Hybrid RAG.

## 1 · Imports

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
print(f"Importing libraries done!")

Importing libraries done!


## 2 · Configuration

In [3]:
SCRIPT_DIR = Path(".").resolve()
load_dotenv(dotenv_path=SCRIPT_DIR.parent / ".env")

# Azure OpenAI
endpoint              = os.getenv("endpoint")
subscription_key      = os.getenv("subscription_key")
embedding_deployment  = os.getenv("embedding_deployment", "text-embedding-3-small")
embedding_api_version = os.getenv("embedding_api_version", "2024-02-01")

# Paths
WIKIPEDIA_DIR = SCRIPT_DIR.parent / "data" / "wikipedia"
VECTORS_DIR   = SCRIPT_DIR.parent / "data" / "wikipedia_vectors"
VECTORS_DIR.mkdir(parents=True, exist_ok=True)

# Qdrant
QDRANT_URL      = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "wikipedia_scientists")
VECTOR_SIZE     = int(os.getenv("VECTOR_SIZE", 1536))   # text-embedding-3-small produces 1536-dimensional vectors

# Chunking  (~1500 chars ≈ ~375 tokens, well within the 8191-token embedding limit)
CHUNK_SIZE    = 1500
CHUNK_OVERLAP = 150
BATCH_SIZE    = 50

print(f"Configuration done!")
print(f"Wikipedia directory : {WIKIPEDIA_DIR}")
print(f"Vectors cache dir   : {VECTORS_DIR}")
print(f"Qdrant URL          : {QDRANT_URL}")
print(f"Collection name     : {COLLECTION_NAME}")
print(f"Vector size         : {VECTOR_SIZE}")


Configuration done!
Wikipedia directory : C:\Repos\GraphRAG_Demo\src\data\wikipedia
Vectors cache dir   : C:\Repos\GraphRAG_Demo\src\data\wikipedia_vectors
Qdrant URL          : http://localhost:6333
Collection name     : wikipedia_scientists
Vector size         : 1536


## 3 · Initialise Clients

In [5]:
openai_client = AzureOpenAI(
    api_version=embedding_api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
    azure_deployment=embedding_deployment,
)

qdrant = QdrantClient(url=QDRANT_URL)

print("Clients initialised.")

Clients initialised.


## 4 · (Re)create Qdrant Collection

In [6]:
existing = [c.name for c in qdrant.get_collections().collections]
if COLLECTION_NAME in existing:
    qdrant.delete_collection(collection_name=COLLECTION_NAME)
    print(f"Deleted existing Qdrant collection '{COLLECTION_NAME}'")

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
)
print(f"Created Qdrant collection '{COLLECTION_NAME}'")

Created Qdrant collection 'wikipedia_scientists'


## 5 · Text Chunking 

In [7]:
def split_into_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    """Split text into overlapping chunks, preferring paragraph boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind("\n\n", start + int(chunk_size * 0.8), end)
            if boundary != -1:
                end = boundary
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap if end - overlap > start else end
    return chunks

## 6 · Embed Articles & Collect Vectors

In [8]:
import json

wikipedia_files = sorted(WIKIPEDIA_DIR.glob("*.txt"))
print(f"Found {len(wikipedia_files)} Wikipedia files to embed.\n")

points   = []
point_id = 0

for file_idx, wiki_file in enumerate(wikipedia_files):
    cache_file = VECTORS_DIR / (wiki_file.stem + ".json")
    title      = wiki_file.stem.replace("_", " ")

    # ── Load from cache if available ──────────────────────────────────────────
    if cache_file.exists():
        cached = json.loads(cache_file.read_text(encoding="utf-8"))
        print(f"  [{file_idx + 1}/{len(wikipedia_files)}] {wiki_file.name} → "
              f"{len(cached)} chunk(s) (loaded from cache)")
        for item in cached:
            points.append(
                PointStruct(
                    id=point_id,
                    vector=item["vector"],
                    payload={
                        "filename":    item["filename"],
                        "title":       item["title"],
                        "chunk_index": item["chunk_index"],
                        "chunk_total": item["chunk_total"],
                        "text":        item["text"],
                    },
                )
            )
            point_id += 1
        continue

    # ── Embed and save to cache ───────────────────────────────────────────────
    text   = wiki_file.read_text(encoding="utf-8").strip()
    chunks = split_into_chunks(text)
    print(f"  [{file_idx + 1}/{len(wikipedia_files)}] {wiki_file.name} → "
          f"{len(chunks)} chunk(s) (embedding…)")

    cache_entries = []
    for chunk_idx, chunk in enumerate(chunks):
        response = openai_client.embeddings.create(
            input=chunk,
            model=embedding_deployment,
        )
        vector = response.data[0].embedding

        entry = {
            "filename":    wiki_file.name,
            "title":       title,
            "chunk_index": chunk_idx,
            "chunk_total": len(chunks),
            "text":        chunk,
            "vector":      vector,
        }
        cache_entries.append(entry)

        points.append(
            PointStruct(
                id=point_id,
                vector=vector,
                payload={k: v for k, v in entry.items() if k != "vector"},
            )
        )
        point_id += 1

    # Save vectors for this file to disk
    cache_file.write_text(json.dumps(cache_entries, ensure_ascii=False), encoding="utf-8")
    print(f"    Saved vectors → {cache_file.name}")

print(f"\nEmbedded {len(points)} chunks in total.")


Found 51 Wikipedia files to embed.

  [1/51] Ada_Lovelace.txt → 32 chunk(s) (embedding…)
    Saved vectors → Ada_Lovelace.json
  [2/51] Al-Khwarizmi.txt → 21 chunk(s) (embedding…)
    Saved vectors → Al-Khwarizmi.json
  [3/51] Alan_Turing.txt → 48 chunk(s) (embedding…)
    Saved vectors → Alan_Turing.json
  [4/51] Albert_Einstein.txt → 68 chunk(s) (embedding…)
    Saved vectors → Albert_Einstein.json
  [5/51] Alexander_Fleming.txt → 22 chunk(s) (embedding…)
    Saved vectors → Alexander_Fleming.json
  [6/51] Antoine_Lavoisier.txt → 36 chunk(s) (embedding…)
    Saved vectors → Antoine_Lavoisier.json
  [7/51] Archimedes.txt → 31 chunk(s) (embedding…)
    Saved vectors → Archimedes.json
  [8/51] Barbara_McClintock.txt → 26 chunk(s) (embedding…)
    Saved vectors → Barbara_McClintock.json
  [9/51] Carl_Sagan.txt → 51 chunk(s) (embedding…)
    Saved vectors → Carl_Sagan.json
  [10/51] Caroline_Herschel.txt → 18 chunk(s) (embedding…)
    Saved vectors → Caroline_Herschel.json
  [11/51] Cecil

## 7 · Upsert Vectors to Qdrant

In [9]:
for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i + BATCH_SIZE]
    qdrant.upsert(collection_name=COLLECTION_NAME, points=batch)
    print(f"  Upserted batch {i // BATCH_SIZE + 1} ({len(batch)} points)")

print(f"\nStored {len(points)} vectors ({point_id} chunks total) in '{COLLECTION_NAME}'.")
print(f"Browse at: {QDRANT_URL}/dashboard")

  Upserted batch 1 (50 points)
  Upserted batch 2 (50 points)
  Upserted batch 3 (50 points)
  Upserted batch 4 (50 points)
  Upserted batch 5 (50 points)
  Upserted batch 6 (50 points)
  Upserted batch 7 (50 points)
  Upserted batch 8 (50 points)
  Upserted batch 9 (50 points)
  Upserted batch 10 (50 points)
  Upserted batch 11 (50 points)
  Upserted batch 12 (50 points)
  Upserted batch 13 (50 points)
  Upserted batch 14 (50 points)
  Upserted batch 15 (50 points)
  Upserted batch 16 (50 points)
  Upserted batch 17 (50 points)
  Upserted batch 18 (50 points)
  Upserted batch 19 (50 points)
  Upserted batch 20 (50 points)
  Upserted batch 21 (50 points)
  Upserted batch 22 (50 points)
  Upserted batch 23 (50 points)
  Upserted batch 24 (50 points)
  Upserted batch 25 (50 points)
  Upserted batch 26 (50 points)
  Upserted batch 27 (50 points)
  Upserted batch 28 (50 points)
  Upserted batch 29 (50 points)
  Upserted batch 30 (50 points)
  Upserted batch 31 (50 points)
  Upserted batch 